# IMPORTATION DES PACKAGES

In [6]:
import os
import time
import requests
import pandas as pd

# REUNION DES BASES DE DONNEES

### On commence par construire une liste de tableaux avec les colonnes pays et date pour chaque base de données en mémoires dans le dossier base_musique

In [ ]:
DOSSIER = "C:/Users/HP/Desktop/ENSAE/2A/PROJET PYTHON/"

tableaux = []      # liste qui va recevoir chaque fichier lu

# Parcourir chaque dossier de pays
for dossier_pays in os.listdir(DOSSIER):
    chemin_pays = os.path.join(DOSSIER, dossier_pays)

    # Parcourir chaque fichier de ce pays
    for nom_fichier in os.listdir(chemin_pays):
        if not nom_fichier.endswith(".csv"):
            continue                                   # on ignore ce qui n'est pas un CSV

        # Lire le fichier
        chemin_fichier = os.path.join(chemin_pays, nom_fichier)
        tableau = pd.read_csv(chemin_fichier)

        # Extraire le pays et la semaine à partir du nom du fichier
        # "regional-ar-weekly-2026-10-01.csv" découpé aux tirets donne :
        # ["regional", "ar", "weekly", "2026", "10", "01.csv"]
        morceaux = nom_fichier.replace(".csv", "").split("-")
        tableau["pays"] = morceaux[1]                               # "ar"
        tableau["semaine"] = morceaux[3] + "-" + morceaux[4] + "-" + morceaux[5]   # "2026-10-01"

        tableaux.append(tableau)

print(len(tableaux), "fichiers lus")

### On construit un dataframe en joingant les différents tableaux

In [ ]:
base_complete = pd.concat(tableaux, ignore_index=True)

# Transformer la colonne semaine en vraie date (utile pour trier et filtrer)
base_complete["semaine"] = pd.to_datetime(base_complete["semaine"])

print(base_complete.shape)
print(base_complete.head())

### On exporte la base final en CSV pour la conserver

In [ ]:
base_complete.to_csv("base_complete.csv", index=False)

# RECUPERATION DE LA NATIONALITE DU CHANTEUR ET DE LA LANGUE DE CHANT

### MusicBrainz veut savoir pourquoi on utilise les données 

In [14]:
HEADERS = {"User-Agent": "ProjetMusique 1"}

### Avant de commencer, nous devons être capable de lire les infos sur le site MusicBrainz. C'est ce que nous permet de faire la fonction qui suit.

In [15]:
def lire(chemin, params):
    
    #Docstring first
    """Ceci est une fonction pour poser une question à MusicBrainz.
    Toutes les recherches passent par cette fonction. 
    Elle attend une seconde (règle de MusicBrainz), pose la question, et réessaie (4 fois de suite) si le serveur ne répond pas."""
    
    adresse = "https://musicbrainz.org/ws/2/" + chemin 
    params["fmt"] = "json"

    for essai in range(5):
        time.sleep(1) #MusicBrainz exige de ne poser qu'au plus une question par seconde 
        try:
            reponse = requests.get(adresse, headers=HEADERS, params=params) 
            #Python ira visiter l'adresse (si c'est Stromae qu'on cherche): https://musicbrainz.org/ws/2/artist?query=Stromae&limit=1&fmt=json
            if reponse.status_code == 200:
                return reponse.json()
        except requests.RequestException:
            pass      #Rien à faire on passe

    return {}                         # échec donc on renvoie un résultat vide

In [16]:
print(lire("artist", {"query": "lookthrusonette", "limit": 1}))

{'created': '2026-10-19T07:19:46.845Z', 'count': 1, 'offset': 0, 'artists': [{'id': 'b2fa9338-e940-4284-8c2b-ed3d65da1423', 'type': 'Person', 'type-id': 'b6e035f4-3ce9-331c-97df-83397230b0df', 'score': 100, 'gender-id': '36d3d30a-839d-3eda-8cb3-29be4384e4a9', 'name': 'lookthrusonette', 'sort-name': 'lookthrusonette', 'gender': 'male', 'area': {'id': '26e0e534-19ea-4645-bfb3-1aa4e83a4046', 'type': 'City', 'type-id': '6fd8f29a-3d0a-32fc-980d-ea697b69da78', 'name': 'Atlanta', 'sort-name': 'Atlanta', 'life-span': {'ended': None}}, 'begin-area': {'id': '26e0e534-19ea-4645-bfb3-1aa4e83a4046', 'type': 'City', 'type-id': '6fd8f29a-3d0a-32fc-980d-ea697b69da78', 'name': 'Atlanta', 'sort-name': 'Atlanta', 'life-span': {'ended': None}}, 'life-span': {'begin': '2009-11-19', 'ended': None}}]}


### Maintenant qu'on sait tout ire sur MusicBrainz, il nous faut maintenant à partir de la fiche d'un artiste, pouvoir récuperer 

In [17]:
def pays_d_un_artiste(fiche):

    """Cette fonction prend la fiche d'un artiste et renvoie le code de son pays d'origine. 
    Mais plusieurs cas sont à consider. Le cas où le pays est directement renseigné dans country, et le cas ou country est vide.
    L'objectif est juste de récupérer le code du pays de l'artiste considéré. La nationalité sera retenue plus tard (car il faut gérer le cas d'un group)"""
    # Cas 1 : le pays est écrit directement sur la fiche
    if fiche.get("country"):
        return fiche["country"]

    # Cas 2 : seule la ville est connue (puisque si le pays était connu, country aurait été automatiquement rempli), on remonte jusqu'au pays
    # Un exemple de ce cas est l'artiste *lookthrusonette*
    zone = fiche.get("area")
    etapes = 0
    while zone and etapes < 6: #etapes<6, on suppose que le max de division pays-ville-quartier... est de 5. Après quoi on déclare le pays introuvable
        etapes = etapes + 1
        fiche_zone = lire("area/" + zone["id"], {"inc": "area-rels"})

        # Si cette zone est un pays, on a trouvé
        if fiche_zone.get("iso-3166-1-codes"): #iso-3166-1-codes est askip le code des pays
            return fiche_zone["iso-3166-1-codes"][0]

        # Sinon, on passe à la zone qui la contient
        zone = None
        for relation in fiche_zone.get("relations", []):
            if relation.get("type") == "part of" and relation.get("direction") == "backward":
                zone = relation["area"] #On récupère la zone et puis on continue dans la boucle pour voir si on peut trouver le pays à partir d'elle
                break

    return ""        # pays introuvable

In [18]:
resultat = lire("artist", {"query": "lookthrusonette", "limit": 1})
fiche = resultat["artists"][0]
print(fiche.get("country"), fiche.get("area")) #On observe qu'on n'a pas le pays. Mais on a la ville
print(pays_d_un_artiste(fiche)) #Notre fonction retrouve bien le code du pays. 
#Pour cette artiste, nous avons constaté qu'il a fallu 4 étapes pour retrouver le pays. 

None {'id': '26e0e534-19ea-4645-bfb3-1aa4e83a4046', 'type': 'City', 'type-id': '6fd8f29a-3d0a-32fc-980d-ea697b69da78', 'name': 'Atlanta', 'sort-name': 'Atlanta', 'life-span': {'ended': None}}
US


### Maintenant que nous somme capable de récupérer le code du pays de n'importe quel artiste, déterminons la nationalité, tout en essaynt de tenir compte des groupes

In [19]:
def nationalite_musicien(nom):
    # 1 : trouver l'artiste
    resultat = lire("artist", {"query": f'artist:"{nom}"', "limit": 1})
    if not resultat.get("artists"):
        # Aucun artiste ne porte ce nom. S'il y a une virgule,
        # c'est peut-être une collaboration : on cherche chaque artiste séparément
        if "," in nom:
            pays_trouves = []
            for un_nom in nom.split(","):
                nationalite = nationalite_musicien(un_nom.strip())
                for pays in nationalite.split(";"):
                    if pays != "" and pays not in pays_trouves: #Pays n'est pas vide et on ne l'a aps déjà dasn la liste de pays
                        pays_trouves.append(pays)
            return ";".join(pays_trouves)
        return ""  # on n'a aucune info sur l'artiste
    id_artiste = resultat["artists"][0]["id"]

    # 2 : ouvrir sa fiche, avec la liste de ses membres si c'est un groupe
    fiche = lire("artist/" + id_artiste, {"inc": "artist-rels"})

    # 3 : une personne a un seul pays
    if fiche.get("type") == "Person":
        return pays_d_un_artiste(fiche)

    # 4 : pour un groupe, on cherche le pays de chaque membre
    pays_trouves = []
    for relation in fiche.get("relations", []):
        if relation.get("type") == "member of band" and relation.get("direction") == "backward":
            fiche_membre = lire("artist/" + relation["artist"]["id"], {})
            pays = pays_d_un_artiste(fiche_membre)
            if pays != "" and pays not in pays_trouves:
                pays_trouves.append(pays)

    # 5 : si aucun membre n'a de pays, on prend celui du groupe
    if len(pays_trouves) == 0:
        return pays_d_un_artiste(fiche)

    return ";".join(pays_trouves)

In [20]:
print(nationalite_musicien("lookthrusonette"))                               # une personne
print(nationalite_musicien("u2"))                                            # un groupe
print(nationalite_musicien("DJ Snake, Selena Gomez, Ozuna, Cardi B"))        # une collaboration

US
IE;US
FR;US;PR


In [21]:
print(nationalite_musicien('Wisin & Yandel'))

PR


In [ ]:
CHEMIN = "artistes.csv"
CHEMIN_FIN = "artistes_fin.csv"

artistes_fin = pd.read_csv(CHEMIN, keep_default_na=False)

for i in range(len(artistes_fin) - 1, -1, -1):      # de la dernière ligne vers la première

    nom = artistes_fin.loc[i, "artist_names"]
    artistes_fin.loc[i, "nationalite"] = nationalite_musicien(nom)

    if i % 20 == 0:
        artistes_fin.to_csv(CHEMIN_FIN, index=False)
        print(i, "lignes restantes")

artistes_fin.to_csv(CHEMIN_FIN, index=False)
print("Terminé !")

9840 lignes restantes
9820 lignes restantes
9800 lignes restantes
9780 lignes restantes
9760 lignes restantes
9740 lignes restantes
9720 lignes restantes
9700 lignes restantes
9680 lignes restantes
9660 lignes restantes
9640 lignes restantes
9620 lignes restantes
9600 lignes restantes
9580 lignes restantes
9560 lignes restantes
9540 lignes restantes
9520 lignes restantes
9500 lignes restantes
9480 lignes restantes
9460 lignes restantes
9440 lignes restantes
9420 lignes restantes
9400 lignes restantes
9380 lignes restantes
9360 lignes restantes
9340 lignes restantes
9320 lignes restantes
9300 lignes restantes
9280 lignes restantes
9260 lignes restantes
9240 lignes restantes
9220 lignes restantes
9200 lignes restantes
9180 lignes restantes
9160 lignes restantes
9140 lignes restantes
9120 lignes restantes
9100 lignes restantes
9080 lignes restantes
9060 lignes restantes
9040 lignes restantes
9020 lignes restantes
9000 lignes restantes
8980 lignes restantes
8960 lignes restantes
8940 ligne

### On connait à présent la nationalité d'un artiste donné. Mais dans laquelle langue a t'il chanté un titre donné ? C'est ce que fait la fonction suivante

In [56]:
def langue_musique(artiste, titre):
    """La fonction prend en entrée l'artiste et le titre et recherche la ou les langue(s) de chanson"""
    # Étape 1 : chercher quelques versions de la chanson
    resultat = lire("recording", {"query": f'recording:"{titre}" AND artist:"{artiste}"', "limit": 5})

    for version in resultat.get("recordings", []):
        # Étape 2 : regarder si cette version est reliée à une oeuvre
        fiche = lire("recording/" + version["id"], {"inc": "work-rels"})

        for relation in fiche.get("relations", []):
            if "work" in relation:
                # Étape 3 : lire la langue de l'oeuvre
                oeuvre = lire("work/" + relation["work"]["id"], {})
                if oeuvre.get("languages"):
                    return ";".join(oeuvre["languages"])

    return ""        # langue introuvable

In [53]:
def langue_musique(artiste, titre):
    # Méthode 1 (rapide) : chercher directement l'oeuvre
    resultat = lire("work", {"query": f'work:"{titre}" AND artist:"{artiste}"', "limit": 1})

    for oeuvre in resultat.get("works", []):
        if oeuvre.get("languages"):
            return ";".join(oeuvre["languages"])

    return ""        # langue introuvable

In [54]:
print(langue_musique("Stromae", "Alors on danse"))

fra


In [57]:
print(langue_musique("DJ Snake, Selena Gomez, Ozuna, Cardi B", "Taki Taki (with Selena Gomez, Ozuna & Cardi B)"))